# Stage 1 - Phases 2-6: forensic branch (F1..F5)

One notebook runs every forensic model. Change `MODEL_NAME` and re-run top to bottom.

Current data policy:
- DLC-2021 supplies ORIGINAL/RERECORDED forensic supervision.
- Available CCD Crash/Normal videos are physical-camera originals only, so any CCD
  rows included in `train.csv` must have Stage 1 label `ORIGINAL`.
- They are used as driving-domain hard negatives; Crash/Normal may be kept only as
  optional `scene_type` metadata.
- Until paired CCD re-recordings exist, checkpoint/threshold selection uses only the
  DLC-2021 portion of `val.csv`.
- No re-splitting or hidden CCD sampling happens inside this notebook.

Recommended forensic order:
`bayar_resnet18 -> chromaticity -> frequency -> lcdf -> cdc`

Native-resolution frames are cropped to forensic patches before any resize, and
evaluation remains video-level Macro-F1.


## 1. Setup

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

# Colab: mount only Google Drive data. The code repository remains under /content.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; Drive mount skipped.')


def find_repo_root() -> Path:
    candidates = [Path('/content/Blackbox-Detection'), Path.cwd()]
    for candidate in candidates:
        current = candidate.resolve()
        while True:
            if (current / 'pyproject.toml').is_file():
                return current
            if current == current.parent:
                break
            current = current.parent
    raise FileNotFoundError(
        'Blackbox-Detection repository not found. Clone/check out your GitHub repo first, '
        'normally at /content/Blackbox-Detection.'
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from blackbox_detection.utils import seed_everything, setup_logger

print('repo :', REPO_ROOT)
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
from blackbox_detection.stage1.dataset import Stage1ForensicDataset, build_dataloader, forensic_batch_adapter
from blackbox_detection.stage1.evaluator import AggregationConfig, Stage1Evaluator, aggregate_unit_predictions, evaluate_predictions, save_predictions
from blackbox_detection.stage1.models import build_stage1_model, count_parameters
from blackbox_detection.stage1.sampling import build_forensic_samplers
from blackbox_detection.stage1.trainer import Stage1Trainer, TrainConfig
from blackbox_detection.stage1.transforms import PatchAugmentConfig, build_forensic_transforms
from blackbox_detection.utils import load_checkpoint

logger = setup_logger("stage1.forensic")

## 2. Paths

In [ ]:
CONFIG_DIR = REPO_ROOT / 'configs' / 'stage1'
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'stage1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Google Drive contains DATASET only. Code stays in /content/Blackbox-Detection.
DATASET_ROOT = Path('/content/drive/MyDrive/Blackbox-Detection/DATASET')
SPLIT_ROOT = DATASET_ROOT / 'stage1_splits'

TRAIN_CSV = SPLIT_ROOT / 'train.csv'
VAL_CSV = SPLIT_ROOT / 'val.csv'
TEST_CSV = SPLIT_ROOT / 'test.csv'  # final holdout; never loaded in training notebooks

# Until physical CCD re-recordings exist, CCD contributes ORIGINAL driving
# hard-negatives to training, but the primary validation metric stays on DLC-2021.
PRIMARY_VAL_DATASET = 'dlc2021'

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f'DATASET_ROOT does not exist: {DATASET_ROOT}\n'
        'Mount Google Drive first or edit DATASET_ROOT.'
    )
if not SPLIT_ROOT.is_dir():
    raise FileNotFoundError(
        f'SPLIT_ROOT does not exist: {SPLIT_ROOT}\n'
        'Ask the data team to place train.csv / val.csv / test.csv there.'
    )

print('repo       :', REPO_ROOT)
print('dataset    :', DATASET_ROOT)
print('split root :', SPLIT_ROOT)
print('train      :', TRAIN_CSV)
print('val        :', VAL_CSV)
print('test       :', TEST_CSV, '(reserved; not loaded)')
print('primary val:', PRIMARY_VAL_DATASET)


## 3. Config

In [ ]:
# One of: bayar_resnet18 | chromaticity | frequency | lcdf | cdc
MODEL_NAME = 'bayar_resnet18'
FORENSIC_CONFIG = yaml.safe_load((CONFIG_DIR / 'forensic.yaml').read_text(encoding='utf-8'))
if MODEL_NAME not in FORENSIC_CONFIG['models']:
    raise ValueError(f'{MODEL_NAME!r} not in forensic config; available: {sorted(FORENSIC_CONFIG["models"])}')

def merge_config(defaults: dict, overrides: dict) -> dict:
    merged = {key: dict(value) for key, value in defaults.items()}
    for section, values in overrides.items():
        merged.setdefault(section, {})
        merged[section] = {**merged[section], **values}
    return merged

CONFIG = merge_config(FORENSIC_CONFIG['defaults'], FORENSIC_CONFIG['models'][MODEL_NAME])
ADAPTER = forensic_batch_adapter()
SEED = int(CONFIG['train']['seed'])
PATCH_SIZE = int(CONFIG['data']['patch_size'])
RUN_DIR = OUTPUT_ROOT / MODEL_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
seed_everything(SEED, deterministic=False)
print(MODEL_NAME, '->', RUN_DIR)
print(json.dumps(CONFIG['model'], indent=2))

## 4. Fixed CSV data

In [ ]:
LABEL_ALIASES = {
    '0': 'ORIGINAL',
    'OR': 'ORIGINAL',
    'ORIGINAL': 'ORIGINAL',
    '1': 'RERECORDED',
    'RE': 'RERECORDED',
    'RERECORDED': 'RERECORDED',
    'RE-RECORDED': 'RERECORDED',
}


def _infer_dataset(path_text: str) -> str:
    lowered = path_text.replace('\\', '/').lower()
    if 'dlc-2021' in lowered or 'dlc2021' in lowered:
        return 'dlc2021'
    if '/ccd/' in f'/{lowered.strip("/")}/' or lowered.startswith('ccd/'):
        return 'ccd'
    return 'unknown'


def _stable_video_id(path_text: str) -> str:
    normalized = path_text.replace('\\', '/')
    stem = Path(normalized).stem
    digest = hashlib.sha1(normalized.encode('utf-8')).hexdigest()[:10]
    return f'{stem}_{digest}'


def _resolve_video_path(value: str) -> str:
    raw = str(value).strip()
    p = Path(raw)
    if p.is_absolute():
        return str(p)

    # CSV paths are relative to DATASET_ROOT. Both forms are accepted:
    #   DLC-2021/or/.../clip.mp4
    #   DATASET/DLC-2021/or/.../clip.mp4
    parts = list(p.parts)
    if parts and parts[0].lower() == 'dataset':
        p = Path(*parts[1:])
    return str((DATASET_ROOT / p).resolve())


def load_stage1_csv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f'CSV not found: {path}')

    frame = pd.read_csv(path).copy()
    required = {'video_path', 'label'}
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f'{path.name} is missing required columns: {missing}')

    raw_paths = frame['video_path'].astype(str).str.strip()
    normalized_labels = frame['label'].astype(str).str.strip().str.upper()

    # Crash/Normal are CCD scene labels, not Stage 1 labels.
    scene_labels = {'CRASH', 'NORMAL'}
    mistaken_scene_labels = sorted(set(normalized_labels) & scene_labels)
    if mistaken_scene_labels:
        raise ValueError(
            f'{path.name}: CCD Crash/Normal must not be used as Stage 1 labels. '
            'Current non-rerecorded CCD videos must have label=ORIGINAL; '
            'store Crash/Normal in an optional scene_type column.'
        )

    mapped = normalized_labels.map(LABEL_ALIASES)
    if mapped.isna().any():
        bad = sorted(normalized_labels[mapped.isna()].unique().tolist())
        raise ValueError(f'{path.name} has unsupported Stage 1 labels: {bad}')
    frame['label'] = mapped

    if 'video_id' not in frame.columns:
        frame['video_id'] = raw_paths.map(_stable_video_id)
    else:
        frame['video_id'] = frame['video_id'].astype(str).str.strip()

    if 'dataset' not in frame.columns:
        frame['dataset'] = raw_paths.map(_infer_dataset)
    else:
        frame['dataset'] = frame['dataset'].astype(str).str.strip().str.lower()
        frame['dataset'] = frame['dataset'].replace(
            {'dlc': 'dlc2021', 'dlc-2021': 'dlc2021', 'ccd_dataset': 'ccd'}
        )

    if (frame['dataset'] == 'unknown').any():
        examples = raw_paths[frame['dataset'] == 'unknown'].head(3).tolist()
        raise ValueError(
            f'{path.name}: could not infer dataset for some rows. '
            f'Add dataset=dlc2021/ccd explicitly. Examples: {examples}'
        )

    frame['video_path'] = raw_paths.map(_resolve_video_path)

    if frame['video_id'].eq('').any():
        raise ValueError(f'{path.name} contains an empty video_id.')
    if frame['video_id'].duplicated().any():
        dup = frame.loc[
            frame['video_id'].duplicated(keep=False), 'video_id'
        ].head(10).tolist()
        raise ValueError(f'{path.name} has duplicated video_id values: {dup}')

    missing_files = [p for p in frame['video_path'] if not Path(p).is_file()]
    if missing_files:
        raise FileNotFoundError(
            f'{path.name}: {len(missing_files)} video file(s) do not exist. '
            f'First examples: {missing_files[:3]}'
        )

    return frame.reset_index(drop=True)


# The team owns the fixed split. These notebooks never create/re-split it.
train_df = load_stage1_csv(TRAIN_CSV)
all_val_df = load_stage1_csv(VAL_CSV)

# Leak check uses the entire team-provided validation CSV before filtering.
video_overlap = set(train_df['video_id']) & set(all_val_df['video_id'])
if video_overlap:
    raise ValueError(f'train/val video_id leakage: {sorted(video_overlap)[:5]}')

if 'source_video_id' in train_df.columns and 'source_video_id' in all_val_df.columns:
    train_sources = set(
        train_df['source_video_id'].dropna().astype(str).str.strip()
    ) - {''}
    val_sources = set(
        all_val_df['source_video_id'].dropna().astype(str).str.strip()
    ) - {''}
    source_overlap = train_sources & val_sources
    if source_overlap:
        raise ValueError(
            f'train/val source_video_id leakage: {sorted(source_overlap)[:5]}'
        )

# Training uses train.csv exactly as supplied: DLC OR/RE plus any CCD ORIGINAL rows.
# No hidden CCD resampling is performed here; the data team controls the ratio in CSV.
if train_df['label'].nunique() < 2:
    raise ValueError('train.csv must contain both ORIGINAL and RERECORDED.')

# Until paired CCD re-recordings exist, use DLC only for model selection/threshold tuning.
val_df = all_val_df[
    all_val_df['dataset'].str.lower().eq(PRIMARY_VAL_DATASET)
].copy().reset_index(drop=True)

if val_df.empty:
    raise ValueError(
        f'val.csv contains no rows for PRIMARY_VAL_DATASET={PRIMARY_VAL_DATASET!r}.'
    )
if val_df['label'].nunique() < 2:
    raise ValueError(
        f'Primary validation ({PRIMARY_VAL_DATASET}) must contain both '
        'ORIGINAL and RERECORDED.'
    )

print('\\nTRAIN dataset x label')
display(pd.crosstab(train_df['dataset'], train_df['label'], margins=True))

print('\\nFULL VAL dataset x label')
display(pd.crosstab(all_val_df['dataset'], all_val_df['label'], margins=True))

print(
    f'Primary validation used for checkpoint/threshold: '
    f'{PRIMARY_VAL_DATASET} ({len(val_df)} videos)'
)

ccd_train = train_df[train_df['dataset'].str.lower().eq('ccd')]
if len(ccd_train):
    ccd_labels = sorted(ccd_train['label'].unique().tolist())
    ccd_share = len(ccd_train) / len(train_df)
    print(
        f'CCD training rows: {len(ccd_train)} '
        f'({ccd_share:.1%} of train), labels={ccd_labels}'
    )
    if ccd_labels == ['ORIGINAL']:
        print(
            'NOTE: current CCD is ORIGINAL-only driving data. It acts as a '
            'driving-domain hard negative, not as a complete OR/RE domain.'
        )
    if ccd_share > 0.5 and 'RERECORDED' not in ccd_labels:
        print(
            'WARNING: ORIGINAL-only CCD is >50% of train. This can encourage '
            'dataset/content shortcuts. Prefer controlling this in train.csv.'
        )

print('test.csv is intentionally reserved for final holdout evaluation.')


### 4.1 Native-resolution frames and patches

In [ ]:
data_config = CONFIG['data']
augmentation_config = CONFIG['augmentation']
patch_augment = PatchAugmentConfig(
    hflip_prob=float(augmentation_config['hflip_prob']),
    vflip_prob=float(augmentation_config['vflip_prob']),
    rot90_prob=float(augmentation_config['rot90_prob']),
    spectral_augment_prob=float(augmentation_config['spectral_augment_prob']),
    spectral_augment_alpha_range=tuple(augmentation_config['spectral_augment_alpha_range']),
    spectral_augment_beta_std=float(augmentation_config['spectral_augment_beta_std']),
    spectral_augment_keep_outside=bool(augmentation_config['spectral_augment_keep_outside']),
)
train_transform, val_transform = build_forensic_transforms(train_config=patch_augment)
train_frame_sampler, train_patch_sampler = build_forensic_samplers(
    train=True, num_frames=int(data_config['train_num_frames']), num_patches=int(data_config['train_num_patches']), patch_size=PATCH_SIZE,
)
val_frame_sampler, val_patch_sampler = build_forensic_samplers(
    train=False, num_frames=int(data_config['val_num_frames']), num_patches=int(data_config['val_num_patches']), patch_size=PATCH_SIZE, val_grid=int(data_config['val_patch_grid']),
)
train_dataset = Stage1ForensicDataset(train_df, frame_sampler=train_frame_sampler, patch_sampler=train_patch_sampler, transform=train_transform, patch_size=PATCH_SIZE, on_error='zero', deterministic=False)
val_dataset = Stage1ForensicDataset(val_df, frame_sampler=val_frame_sampler, patch_sampler=val_patch_sampler, transform=val_transform, patch_size=PATCH_SIZE, on_error='zero', deterministic=True)
train_loader = build_dataloader(train_dataset, batch_size=int(data_config['batch_size']), shuffle=True, num_workers=int(data_config['num_workers']), seed=SEED, drop_last=True)
val_loader = build_dataloader(val_dataset, batch_size=int(data_config['val_batch_size']), shuffle=False, num_workers=int(data_config['num_workers']), seed=SEED)
print('patches per training video:', train_dataset.num_units)
print('patches per validation video:', val_dataset.num_units)
batch = next(iter(train_loader))
adapted = ADAPTER.unpack(batch, 'cpu')
print('patch batch:', tuple(batch['patches'].shape), '-> units:', tuple(adapted.inputs.shape))

## 5. Model

In [ ]:
model = build_stage1_model(
    MODEL_NAME,
    finetune_mode=CONFIG['model']['finetune_mode'],
    unfreeze_last_n=int(CONFIG['model']['unfreeze_last_n']),
    **CONFIG['model']['params'],
)
print('blocks:', len(model.blocks), '| feature dim:', model.feature_dim)
print('parameters:', count_parameters(model))
print('preprocessing:', dict(model.preprocessing()))
with torch.no_grad():
    probe = model(adapted.inputs[:2])
print('logits:', tuple(probe.shape))

## 6. Training

In [ ]:
train_config = CONFIG['train']
trainer_config = TrainConfig(
    epochs=int(train_config['epochs']),
    learning_rate=float(train_config['learning_rate']),
    head_learning_rate=(
        float(train_config['head_learning_rate'])
        if train_config.get('head_learning_rate') is not None
        else None
    ),
    weight_decay=float(train_config['weight_decay']),
    warmup_ratio=float(train_config['warmup_ratio']),
    grad_accum_steps=int(train_config['grad_accum_steps']),
    max_grad_norm=float(train_config['max_grad_norm']),
    amp=bool(train_config['amp']),
    label_smoothing=float(train_config.get('label_smoothing', 0.0)),
    early_stopping_patience=int(train_config['early_stopping_patience']),
    eval_every=int(train_config['eval_every']),
    seed=SEED,
    output_dir=RUN_DIR,
    model_name=MODEL_NAME,
    wandb_enabled=False,
)

trainer = Stage1Trainer(
    model,
    trainer_config,
    adapter=ADAPTER,
    aggregation=AggregationConfig(
        frame_method=CONFIG['evaluation']['aggregation']['frame_method'],
        video_method=CONFIG['evaluation']['aggregation']['video_method'],
    ),
    model_config={'name': MODEL_NAME, 'params': CONFIG['model']['params']},
)

print('device:', trainer.device, '| amp:', trainer.amp)
outcome = trainer.fit(train_loader, val_loader)
print(
    f'best epoch {outcome.best_epoch}: Macro-F1 {outcome.best_macro_f1:.4f} '
    f'at threshold {outcome.best_threshold:.3f}'
)

## 7. Validation

In [ ]:
load_checkpoint(RUN_DIR / 'best.pt', model=model, map_location=trainer.device, restore_rng_state=False)
evaluator = Stage1Evaluator(model, ADAPTER, device=trainer.device, amp=trainer.amp, aggregation=trainer.aggregation)
result, units = evaluator.evaluate(val_loader, return_units=True)
print(f'Macro-F1            : {result.macro_f1:.4f}')
print(f'Macro-F1 @ thr 0.5  : {result.macro_f1_at_default:.4f}')
print(f'optimal threshold   : {result.threshold:.4f}')
print(f'class-wise F1       : {result.per_class_f1}')
print(f'per-dataset Macro-F1: {result.dataset_scores}')
print(f'videos              : {result.num_videos} ({result.num_invalid_videos} with decode problems)')

rows = []
for video_method in ('mean', 'median', 'trimmed_mean', 'logit_mean', 'max'):
    aggregated = aggregate_unit_predictions(units, aggregation=AggregationConfig(frame_method='mean', video_method=video_method))
    scored = evaluate_predictions(aggregated)
    rows.append({'frame': 'mean', 'video': video_method, 'macro_f1': scored.macro_f1, 'threshold': scored.threshold})
display(pd.DataFrame(rows))

## 8. Save

In [ ]:
save_predictions(result.predictions, RUN_DIR / 'val_predictions.csv')
outcome.history.to_csv(RUN_DIR / 'history.csv', index=False)
summary = {
    'model_name': MODEL_NAME,
    'val_macro_f1': float(result.macro_f1),
    'val_macro_f1_at_0.5': float(result.macro_f1_at_default),
    'best_threshold': float(result.threshold),
    'per_class_f1': result.per_class_f1,
    'best_epoch': int(outcome.best_epoch),
    'num_val_videos': int(result.num_videos),
    'preprocessing': dict(model.preprocessing()),
}
(RUN_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
print('saved to:', RUN_DIR)